# F07 - Supervised and unsupervised learning

**Hardware:** CPU-only · **Time:** 2-3 hours · **Prerequisite:** F06

Supervised learning uses labeled examples to predict a target. Unsupervised
learning looks for structure without a target. This notebook applies both to a
synthetic customer table. The latent segment is withheld during clustering and
used only afterward to evaluate what the algorithm discovered.

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
print("pandas", pd.__version__)

pandas 2.3.3


## 1. A table with hidden structure

Each row describes a customer by spend, visits, age, and online share. The
`latent_segment` column is generated from three groups but is normally hidden
from the analyst. We keep it for evaluation only.

In [2]:
def make_customers(n: int = 420, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    latent = rng.choice([0, 1, 2], n, p=[0.4, 0.35, 0.25])
    spend = np.choose(latent, [180, 650, 1200]) + rng.normal(0, 60, n)
    visits = np.choose(latent, [2.0, 8.0, 18.0]) + rng.normal(0, 1.5, n)
    age = np.choose(latent, [24, 38, 52]) + rng.normal(0, 4, n)
    online = np.choose(latent, [0.8, 0.45, 0.2]) + rng.normal(0, 0.08, n)
    return pd.DataFrame(
        {
            "annual_spend": spend.round(2).clip(20, None),
            "visits_per_month": visits.round(1).clip(0.2, None),
            "age": age.round(0).clip(18, 80).astype(int),
            "online_share": online.round(3).clip(0, 1),
            "latent_segment": latent,
        }
    )


customers = make_customers()
features = ["annual_spend", "visits_per_month", "age", "online_share"]
X = StandardScaler().fit_transform(customers[features])
print("shape:", X.shape)
print(customers[features].describe().round(2).to_string())
assert X.shape == (420, 4)
assert np.allclose(X.mean(axis=0), 0, atol=1e-9)

shape: (420, 4)
       annual_spend  visits_per_month     age  online_share
count        420.00            420.00  420.00        420.00
mean         603.39              8.24   35.95          0.51
std          411.52              6.60   11.89          0.26
min           26.12              0.20   18.00          0.00
25%          194.22              2.50   26.00          0.29
50%          620.23              7.45   36.00          0.48
75%         1076.85             16.00   46.00          0.76
max         1319.84             22.80   61.00          0.99


## 2. Choose the number of clusters

K-means minimizes within-cluster distance. More clusters always reduce inertia,
so inertia alone cannot choose `k`. The silhouette score rewards clusters that
are compact and well separated. Inspect both and prefer the simplest defensible
solution.

In [3]:
rows = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    rows.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(X, model.labels_),
        }
    )
scores = pd.DataFrame(rows)
print(scores.round(3).to_string(index=False))
assert scores.loc[scores["silhouette"].idxmax(), "k"] == 3

 k  inertia  silhouette
 2  526.384       0.619
 3  119.248       0.707
 4  107.006       0.552
 5   95.310       0.363
 6   85.521       0.246


## 3. Fit the chosen model and profile the clusters

The algorithm has no access to `latent_segment`. After fitting, compare cluster
means on the input features. A cluster is a description, not a causal claim.

In [4]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(X)
customers["cluster"] = kmeans.labels_
profile = customers.groupby("cluster")[features].mean().round(2)
print(profile.to_string())
print("cluster sizes:", customers["cluster"].value_counts().sort_index().to_dict())
assert customers["cluster"].nunique() == 3

         annual_spend  visits_per_month    age  online_share
cluster                                                     
0              651.01              7.92  37.38          0.44
1             1193.45             18.16  52.05          0.19
2              173.96              1.99  24.13          0.79


cluster sizes: {0: 143, 1: 110, 2: 167}


## 4. Evaluate against the withheld labels

Adjusted Rand Index (ARI) compares two label assignments; 1.0 is a perfect
match and 0.0 is chance-level agreement. A high ARI here is expected because the
generator created well-separated groups. Real data is usually messier.

In [5]:
ari = adjusted_rand_score(customers["latent_segment"], customers["cluster"])
print("adjusted Rand index:", round(ari, 3))
assert ari > 0.5

adjusted Rand index: 1.0


## 5. Visualize with PCA

Principal component analysis (PCA) projects the four standardized features onto
two directions with the most variance. The plot is a summary, not proof: two
points that look close in 2D may be far apart in 4D.

In [6]:
pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(X)
fig, ax = plt.subplots(figsize=(6, 4.5))
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=customers["cluster"], cmap="viridis", s=22, alpha=0.75)
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("Customer clusters in PCA space")
fig.colorbar(scatter, label="cluster")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "f07_clusters.png", bbox_inches="tight")
plt.close(fig)
print("explained variance:", np.round(pca.explained_variance_ratio_, 3))
assert (OUTPUT_DIR / "f07_clusters.png").exists()

explained variance: [0.938 0.03 ]


## 6. Supervised versus unsupervised

A supervised model needs a target. If segment labels existed for a sample, a
classifier could predict them for new customers. The classifier below is trained
on a labeled subset (simulating an expert-labeled sample) and evaluated on the
rest. Compare its accuracy with the clustering ARI: they answer different
questions.

| Question | Method | Evidence |
| --- | --- | --- |
| What groups exist? | Clustering | silhouette, profiles, stability |
| Which group will this customer join? | Classification | accuracy, per-class recall |
| How well do groups match known labels? | ARI | agreement, not accuracy |

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, customers["latent_segment"], test_size=0.3, random_state=SEED, stratify=customers["latent_segment"]
)
classifier = LogisticRegression(max_iter=2000, random_state=SEED).fit(X_train, y_train)
accuracy = classifier.score(X_test, y_test)
print("supervised accuracy with labels:", round(accuracy, 3))
assert accuracy > 0.9

supervised accuracy with labels: 1.0


## Check your work

- [ ] You can distinguish a supervised target from an unsupervised structure.
- [ ] You chose `k` using both inertia and silhouette.
- [ ] You described each cluster with feature means.
- [ ] You can explain why ARI is not the same as classification accuracy.
- [ ] `outputs/f07_clusters.png` exists.

**Next:** F08 - Regression and classification.